###  Training a simple wake word model with Picovoice Porcupine_test2




### WAKEWORD DETECTION

In [1]:
import pvporcupine
import sounddevice as sd
import struct

# Recording Setup 
INPUT_DEVICE_ID = 8
CHANNELS = 1

ACCESS_KEY = "1UQRnlg6WE6MbpJO1jX135lftC3GCuJE8+sL7hIVqtiQwLfkfjfrhg=="
#Audio_Response_path = 

# Load the custom wake word model
porcupine = pvporcupine.create(
    access_key=ACCESS_KEY,
    keyword_paths=[r"C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\3- Wake_word\Trained_Model\Hey-Robot_en_windows_v3_0_0.ppn"]
)

def audio_callback(indata, frames, time, status):
    if status:
        print(status)

    pcm = (indata[:, 0] * 32767).astype("int16")

    # Process the frame
    keyword_index = porcupine.process(pcm)
    if keyword_index >= 0:
        print("Wake word detected!")

# ---- IMPORTANT FIX ----
FRAME_LENGTH = porcupine.frame_length

with sd.InputStream(
    device=INPUT_DEVICE_ID,
    channels=CHANNELS,
    samplerate=porcupine.sample_rate,   # Porcupine requires 16 kHz!
    blocksize=FRAME_LENGTH,              # <<< FIXED HERE
    dtype='float32',
    callback=audio_callback
):
    print("Listening for wake word...")
    try:
        while True:
            pass
    except KeyboardInterrupt:
        print("Exiting...")

porcupine.delete()


Listening for wake word...
Wake word detected!
Exiting...


### Audio Feedback 


In [4]:
import pvporcupine
import sounddevice as sd
import soundfile as sf
import threading

# ---- Load the YES sound ----
YES_PATH = r"C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\3- Wake_word\Verbal_Wake_Response\yes-masterrr.wav"
YES_audio, YES_sr = sf.read(YES_PATH, dtype='float32')

# Flag to signal main loop
play_YES_flag = False

# Porcupine setup
ACCESS_KEY = "1UQRnlg6WE6MbpJO1jX135lftC3GCuJE8+sL7hIVqtiQwLfkfjfrhg=="

porcupine = pvporcupine.create(
    access_key=ACCESS_KEY,
    keyword_paths=[r"C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\3- Wake_word\Trained_Model\Hey-Robot_en_windows_v3_0_0.ppn"]
)

FRAME_LENGTH = porcupine.frame_length


def play_YES():
    """Play the YES sound in a separate thread."""
    sd.play(YES_audio, YES_sr)
    sd.wait()


def audio_callback(indata, frames, time, status):
    global play_YES_flag

    if status:
        print(status)

    pcm = (indata[:, 0] * 32767).astype("int16")
    keyword_index = porcupine.process(pcm)

    if keyword_index >= 0:
        print("Wake word detected!")
        play_YES_flag = True  # Signal main loop


# ---- Audio stream ----
with sd.InputStream(
    device=8,
    channels=1,
    samplerate=porcupine.sample_rate,
    blocksize=FRAME_LENGTH,
    dtype='float32',
    callback=audio_callback
):
    print("Listening for wake word...")
    try:
        while True:
            if play_YES_flag:
                play_YES_flag = False

                # Start playing in a background thread
                threading.Thread(target=play_YES, daemon=True).start()

    except KeyboardInterrupt:
        print("Exiting...")

porcupine.delete()


Listening for wake word...
Wake word detected!
Exiting...


### INPUT/OUPUT controllability

In [ ]:
#Finding ID of Audio Devices
import sounddevice as sd 
import numpy as np
import wave
print (sd.query_devices())
print (sd.default.device)

INPUT_DEVICE_ID = 8
OUTPUT_DEVICE_ID = 4

device_info_input = sd.query_devices(INPUT_DEVICE_ID) 
print ("Input Sample Rate is: ",device_info_input['default_samplerate']) # Sample Rate of Input Device

device_info_output = sd.query_devices(OUTPUT_DEVICE_ID ) 
print ("Output Sample Rate is: ",device_info_output['default_samplerate']) # Sample Rate of Output Device


In [5]:
import pvporcupine
import sounddevice as sd
import soundfile as sf
import threading

# Choosing Input/Output Devices
INPUT_DEVICE_ID = 8
OUTPUT_DEVICE_ID = 4

# Load the YES sound
YES_PATH = r"C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\3- Wake_word\Verbal_Wake_Response\yes-masterrr.wav"
YES_audio, YES_sr = sf.read(YES_PATH, dtype='float32')

# Flag to signal main loop
play_YES_flag = False

# Porcupine setup
ACCESS_KEY = "1UQRnlg6WE6MbpJO1jX135lftC3GCuJE8+sL7hIVqtiQwLfkfjfrhg=="

porcupine = pvporcupine.create(
    access_key=ACCESS_KEY,
    keyword_paths=[r"C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\3- Wake_word\Trained_Model\Hey-Robot_en_windows_v3_0_0.ppn"]
)

FRAME_LENGTH = porcupine.frame_length


def play_YES():
    """Play the YES sound in a separate thread."""
    sd.play(YES_audio, YES_sr, device=OUTPUT_DEVICE_ID)
    sd.wait()


def audio_callback(indata, frames, time, status):
    global play_YES_flag

    if status:
        print(status)

    pcm = (indata[:, 0] * 32767).astype("int16")
    keyword_index = porcupine.process(pcm)

    if keyword_index >= 0:
        print("Wake word detected!")
        play_YES_flag = True  # Signal main loop


# ---- Audio stream ----
with sd.InputStream(
    device=INPUT_DEVICE_ID,
    channels=1,
    samplerate=porcupine.sample_rate,
    blocksize=FRAME_LENGTH,
    dtype='float32',
    callback=audio_callback
):
    print("Listening for wake word...")
    try:
        while True:
            if play_YES_flag:
                play_YES_flag = False

                # Start playing in a background thread
                threading.Thread(target=play_YES, daemon=True).start()

    except KeyboardInterrupt:
        print("Exiting...")

porcupine.delete()


Listening for wake word...
Wake word detected!
Exiting...
